# 건물은 학생을 만드는가 — 분석 노트북

단과대학 건물 인프라 차이와 학생의 만족도, 삶의 질 (혼합방법 연구)

**데이터 리터러시 원칙**
1. 원자료 없이 숫자를 임의 생성하지 않는다 — 본 노트북은 `data/survey_data.csv`에 기록된 *보고된 요약 통계치*만 사용한다.
2. 결측치를 조용히 평균으로 채우거나 이상치를 기계적으로 삭제하지 않는다.
3. 요약 시 평균만 제시하지 않고, 가능하면 중앙값·n·표준편차를 함께 보고한다.
4. '분포 개형 개념도'는 보고된 Hedges' g에 기반한 도식이며 실측 분포가 아님을 반드시 명시한다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

import platform
if platform.system() == "Windows":
    mpl.rcParams["font.family"] = "Malgun Gothic"
else:
    mpl.rcParams["font.family"] = "Apple SD Gothic Neo"
mpl.rcParams["axes.unicode_minus"] = False

plt.style.use("seaborn-v0_8-darkgrid")
sns = None
try:
    import seaborn as sns
    sns.set_palette("muted")
except ImportError:
    pass

df = pd.read_csv("../data/survey_data.csv", comment="#")
df

## 1. 격차의 실재 — 실측 종합 상대 점수

17개 실측 지표를 표준화한 종합 상대 점수: **생물관 0.80 vs 경제통상관 1.20** (기준 1.00)

In [ ]:
score = df[df.metric == "composite_relative_score"].set_index("group")["value"]

colors = ["#E8A33D", "#5865F2"]
fig, ax = plt.subplots(figsize=(8, 5), facecolor="#1E1F22")
ax.set_facecolor("#1E1F22")
bars = ax.barh(
    ["경제통상관", "생물관"],
    [score["경제통상관"], score["생물관"]],
    color=[colors[1], colors[0]], height=0.5,
)
ax.axvline(1.0, color="#B5BAC1", linestyle=(0, (5, 5)), linewidth=1.2)
ax.text(1.02, 0.02, "기준 1.00", color="#80848E", fontsize=10, va="bottom")
for b in bars:
    ax.text(b.get_width() + 0.015, b.get_y() + b.get_height() / 2,
            f"{b.get_width():.2f}", va="center", color="#F2F3F5", fontsize=14, fontweight=600)
ax.set_xlim(0, 1.45)
ax.tick_params(colors="#9AA0A6", labelsize=12)
ax.spines[["top", "right"]].set_visible(False)
for s in ax.spines.values():
    s.set_color("#3A3D42")
ax.set_title("실측 종합 상대 점수 (n = 17개 지표)", color="#F2F3F5", fontsize=14, fontweight=600)
plt.tight_layout()
plt.show()

## 2. 효과 크기 요약 — Hedges' g 숲그림(forest plot)

해석 기준(Cohen의 관례): 0.2 = 작음, 0.5 = 중간, 0.8 = 큼

In [ ]:
g = df[df.metric == "hedges_g"].copy()
g["g"] = g["value"]
g = g.sort_values("g")

fig, ax = plt.subplots(figsize=(9, 5), facecolor="#1E1F22")
ax.set_facecolor("#1E1F22")
for lo, hi, c in [(0.2, 0.5, "#FEE75C"), (0.5, 0.8, "#E8A33D"), (0.8, 2.6, "#ED4245")]:
    ax.axvspan(lo, hi, color=c, alpha=0.07)
for x in (0, 0.2, 0.5, 0.8):
    ax.axvline(x, color="#3A3D42", linewidth=0.8, linestyle=(0, (3, 4)))
ax.scatter(g["g"], np.arange(len(g)), s=90, color="#2997FF", zorder=3)
for y, (_, row) in enumerate(g.iterrows()):
    ax.text(row["g"] + 0.05, y, f"g = {row['g']:.2f}", va="center", color="#F2F3F5", fontsize=11, fontweight=600)
ax.set_yticks(np.arange(len(g)))
ax.set_yticklabels(g["group"], color="#B5BAC1", fontsize=11)
ax.set_xlim(-0.2, 2.7)
ax.tick_params(colors="#9AA0A6")
ax.set_xlabel("Hedges' g", color="#9AA0A6")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("설문 88명 — 구인별 효과 크기", color="#F2F3F5", fontsize=14, fontweight=600)
plt.tight_layout()
plt.show()

## 3. 평균의 함정 — 보고 원칙

- 시설 만족도: g = 1.41 (큰 효과)
- 대학생활 만족: g = 0.15 / 삶의 만족: g = 0.36 (평균적으로는 유의한 차이 없음)
- 수업 외 체류 시간: g = 0.51 — 학생이 불편한 건물을 떠나면서 격차가 평균에서 상쇄됨

평균만 보고하면 격차가 '없는 것처럼' 보인다. 분포와 집단 구분이 필요하다.

In [ ]:
takeaways = {
    "시설 만족도": 1.41,
    "우리 건물 소속감": 2.55,
    "삶의 만족(2~4학년)": 0.72,
    "수업 외 체류 시간": 0.51,
    "삶의 만족": 0.36,
    "대학생활 만족": 0.15,
}
fig, ax = plt.subplots(figsize=(9, 4.5), facecolor="#1E1F22")
ax.set_facecolor("#1E1F22")
labels = list(takeaways.keys())[::-1]
vals = [takeaways[k] for k in labels]
cols = ["#ED4245" if v >= 0.8 else "#E8A33D" if v >= 0.5 else "#57F287" for v in vals]
ax.barh(labels, vals, color=cols, height=0.55)
ax.axvline(0.8, color="#ED4245", linewidth=1, linestyle=(0, (4, 4)), alpha=0.7)
ax.text(0.83, -0.6, "large(0.8)", color="#ED4245", fontsize=9, alpha=0.8)
for i, v in enumerate(vals):
    ax.text(v + 0.03, i, f"{v:.2f}", va="center", color="#F2F3F5", fontweight=600)
ax.set_xlim(0, 2.9)
ax.tick_params(colors="#9AA0A6")
ax.set_yticklabels(labels, color="#B5BAC1")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("효과 크기 한눈에 보기 (n = 88)", color="#F2F3F5", fontsize=14, fontweight=600)
plt.tight_layout()
plt.show()

## 4. 분포 쪼개기 — 이봉 분포 개념도

**경고: 아래 그림은 보고된 Hedges' g 값에 기반해 재구성한 분포 개형 개념도다. 실측 분포가 아니다.**

논리: 전체(생물관) 삶의 만족 분포는 사실 두 집단(1학년 / 옛 사회관을 경험한 2~4학년)의 혼합이다.
2~4학년의 삶의 만족만 경제통상관 대비 g = 0.72 하락하고, 1학년은 차이가 없다.

In [ ]:
rng = np.random.default_rng(42)
N = 4000
x = np.linspace(-4, 4, 300)
norm = lambda m, s: np.exp(-0.5 * ((x - m) / s) ** 2) / (s * np.sqrt(2 * np.pi))

fig, axes = plt.subplots(1, 3, figsize=(15, 4), facecolor="#1E1F22")
fig.suptitle("분포 개형 개념도 — 보고된 g 값 기반 재구성 (실측 분포 아님)", color="#F2F3F5", fontsize=13, fontweight=600)

panels = [
    ("① 전체 비교 (겉보기)", {"생물관(전체)": (0.36, 1.0, "#E8A33D"), "경제통상관": (0.0, 1.0, "#5865F2")}),
    ("② 1학년 — 차이 없음 (g ≈ 0)", {"생물관 1학년": (0.0, 1.0, "#64B5CD"), "경제통상관 1학년": (0.02, 1.0, "#5865F2")}),
    ("③ 2~4학년 — 하락 (g = 0.72)", {"생물관 2~4학년": (0.72, 1.0, "#E05D6E"), "경제통상관 2~4학년": (0.0, 1.0, "#5865F2")}),
]
for ax, (title, dists) in zip(axes, panels):
    ax.set_facecolor("#1E1F22")
    for name, (m, s, c) in dists.items():
        ax.plot(x, norm(m, s), color=c, linewidth=2.4, label=name)
        ax.fill_between(x, norm(m, s), color=c, alpha=0.18)
    ax.set_title(title, color="#B5BAC1", fontsize=12, fontweight=600)
    ax.legend(loc="upper left", fontsize=9, labelcolor="#B5BAC1", framealpha=0.2, edgecolor="#3A3D42")
    ax.set_yticks([])
    ax.tick_params(colors="#80848E")
    ax.spines[["top", "right", "left"]].set_visible(False)
    for s in ax.spines.values():
        s.set_color("#3A3D42")
plt.tight_layout()
plt.show()

## 5. 원자료 입수 시 파이프라인 (TODO)

1. 설문 88명 원자료가 확보되면 `data/survey_data.csv`를 원자료로 교체한다.
2. 집단 간 비교는 **Hedges' g**로 통일: `pip install pingouin` → `pg.compute_effsize(x, y, eftype="hedges")`
3. 요약 보고 시 `df.groupby("집단").agg(n=("값", "size"), mean=("값", "mean"), median=("값", "median"), sd=("값", "std"))` 형태로 평균·중앙값·n·SD를 함께 출력한다.
4. 결측치가 있으면 삭제하지 말고 `isna().sum()`으로 보고 후 문항 특성에 맞는 처리 방침을 문서화한다.
5. 재현된 수치가 발표 원고의 보고값과 일치하는지 교차 확인한다.